In [ ]:
# =============================================================================
# 神经风格迁移（Neural Style Transfer）
# =============================================================================
# 神经风格迁移：将一张内容图像的内容与一张风格图像的艺术风格融合，生成新的图像
# 核心思想：利用预训练CNN提取特征，内容特征保留物体结构，风格特征捕获纹理/色彩
# 原始论文："A Neural Algorithm of Artistic Style" (Gatys et al., 2015)

%matplotlib inline
import torch
import torchvision
from torch import nn
from d2l import torch as d2l

# 设置图像显示尺寸
d2l.set_figsize()

# =============================================================================
# 步骤1：加载内容图像和风格图像
# =============================================================================
# 内容图像：提供物体的结构和内容（如山峰、建筑）
content_img = d2l.Image.open('../img/rainier.jpg')
d2l.plt.figure()  # 创建新Figure，避免图像重叠显示
d2l.plt.imshow(content_img);

# 风格图像：提供艺术风格（如颜色、纹理、笔触）
style_img = d2l.Image.open('../img/autumn-oak.jpg')
d2l.plt.figure()  # 关键：再次创建新Figure，确保两张图分开显示
d2l.plt.imshow(style_img);


# =============================================================================
# 步骤2：定义图像预处理和后处理
# =============================================================================
# 使用ImageNet的均值和标准差进行标准化（与预训练VGG保持一致）
rgb_mean = torch.tensor([0.485, 0.456, 0.406])
rgb_std = torch.tensor([0.229, 0.224, 0.225])

def preprocess(img, image_shape):
    """预处理：将PIL图像转换为模型输入格式
    
    处理流程：
    1. Resize：调整图像尺寸
    2. ToTensor：转为tensor并归一化到[0,1]
    3. Normalize：使用ImageNet均值方差标准化
    4. unsqueeze(0)：添加batch维度
    """
    transforms = torchvision.transforms.Compose([
        torchvision.transforms.Resize(image_shape),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(mean=rgb_mean, std=rgb_std)])
    return transforms(img).unsqueeze(0)

def postprocess(img):
    """后处理：将模型输出转换回可显示的PIL图像
    
    逆操作预处理：
    1. [0]移除batch维度
    2. permute(1,2,0)将(C,H,W)转为(H,W,C)便于逐像素操作
    3. *rgb_std + rgb_mean：反标准化
    4. clamp(0,1)：裁剪到有效像素范围
    5. 转回PIL图像格式
    """
    img = img[0].to(rgb_std.device)
    # 反标准化并裁剪到[0,1]范围
    img = torch.clamp(img.permute(1, 2, 0) * rgb_std + rgb_mean, 0, 1)
    # 转回(C,H,W)格式用于ToPILImage
    return torchvision.transforms.ToPILImage()(img.permute(2, 0, 1))


# =============================================================================
# 步骤3：加载预训练VGG19网络
# =============================================================================
# VGG19是深度卷积网络，其特征层能捕获从低级（边缘、纹理）到高级（物体部件）的特征
# pretrained=True加载在ImageNet上预训练的权重
pretrained_net = torchvision.models.vgg19(pretrained=True)

# =============================================================================
# 步骤4：选择内容层和风格层
# =============================================================================
# 内容层：较深层的特征包含更多语义信息，较少纹理细节
# 选择第25层（conv4_2）：在内容表示和计算效率间取得平衡
content_layers = [25]

# 风格层：多个浅层和深层组合，捕获多尺度风格特征
# [0, 5, 10, 19, 28] 对应 conv1_1, conv2_1, conv3_1, conv4_1, conv5_1
# 浅层捕获低级纹理，深层捕获高级风格模式
style_layers = [0, 5, 10, 19, 28]

# 构建特征提取网络：只保留到最深使用层为止的特征层
net = nn.Sequential(*[pretrained_net.features[i] for i in
                      range(max(content_layers + style_layers) + 1)])


# =============================================================================
# 步骤5：定义特征提取函数
# =============================================================================
def extract_features(X, content_layers, style_layers):
    """提取内容和风格特征
    
    参数:
        X: 输入图像tensor
        content_layers: 内容层索引列表
        style_layers: 风格层索引列表
    返回:
        contents: 内容层特征列表
        styles: 风格层特征列表
    """
    contents = []
    styles = []
    for i in range(len(net)):
        X = net[i](X)  # 逐层前向传播
        if i in style_layers:
            styles.append(X)  # 记录风格层特征
        if i in content_layers:
            contents.append(X)  # 记录内容层特征
    return contents, styles


def get_contents(image_shape, device):
    """获取内容图像的特征"""
    content_X = preprocess(content_img, image_shape).to(device)
    contents_Y, _ = extract_features(content_X, content_layers, style_layers)
    return content_X, contents_Y


def get_styles(image_shape, device):
    """获取风格图像的特征"""
    style_X = preprocess(style_img, image_shape).to(device)
    _, styles_Y = extract_features(style_X, content_layers, style_layers)
    return style_X, styles_Y


# =============================================================================
# 步骤6：定义损失函数
# =============================================================================
def content_loss(Y_hat, Y):
    """内容损失：合成图像与内容图像在内容层上的MSE
    
    参数:
        Y_hat: 合成图像的内容特征
        Y: 内容图像的内容特征
    返回:
        均方误差损失
    
    Y.detach()：从计算图中分离目标，避免对内容图像计算梯度
    """
    return torch.square(Y_hat - Y.detach()).mean()


def gram(X):
    """计算Gram矩阵（用于捕获风格特征）
    
    Gram矩阵计算特征图通道间的相关性，捕获纹理信息而不保留空间结构
    
    参数:
        X: 特征图，shape为(batch, channels, h, w)
    返回:
        Gram矩阵，shape为(channels, channels)
    
    计算：G[i,j] = sum(X[i] * X[j]) / (channels * h * w)
    """
    num_channels, n = X.shape[1], X.numel() // X.shape[1]
    # reshape为(channels, h*w)，将空间维度展平
    X = X.reshape((num_channels, n))
    # 计算Gram矩阵并归一化，防止数值过大
    return torch.matmul(X, X.T) / (num_channels * n)


def style_loss(Y_hat, gram_Y):
    """风格损失：合成图像与风格图像的Gram矩阵之间的MSE
    
    参数:
        Y_hat: 合成图像的风格特征
        gram_Y: 风格图像风格的Gram矩阵（预计算）
    """
    return torch.square(gram(Y_hat) - gram_Y.detach()).mean()


def tv_loss(Y_hat):
    """全变分损失（Total Variation Loss）：平滑正则化
    
    原理：惩罚相邻像素间的差异，使生成图像更平滑，减少噪声
    
    计算：
    - 水平方向差异：|Y[:,:,1:,:] - Y[:,:,:-1,:]|
    - 垂直方向差异：|Y[:,:,:,1:] - Y[:,:,:,:-1]|
    """
    return 0.5 * (torch.abs(Y_hat[:, :, 1:, :] - Y_hat[:, :, :-1, :]).mean() +
                  torch.abs(Y_hat[:, :, :, 1:] - Y_hat[:, :, :, :-1]).mean())


# =============================================================================
# 步骤7：定义损失权重和计算函数
# =============================================================================
# 权重超参数，控制各损失的相对重要性
content_weight, style_weight, tv_weight = 1, 1e3, 10
# content_weight=1：内容保真度
# style_weight=1000：风格强度（需要较大值才能体现风格）
# tv_weight=10：平滑程度

def compute_loss(X, contents_Y_hat, styles_Y_hat, contents_Y, styles_Y_gram):
    """计算总损失
    
    参数:
        X: 合成图像
        contents_Y_hat: 合成图像的内容特征
        styles_Y_hat: 合成图像的风格特征
        contents_Y: 内容图像的特征（目标）
        styles_Y_gram: 风格图像的Gram矩阵列表（目标）
    返回:
        contents_l: 各层内容损失列表
        styles_l: 各层风格损失列表
        tv_l: 全变分损失
        l: 总损失
    """
    # 计算加权内容损失
    contents_l = [content_loss(Y_hat, Y) * content_weight for Y_hat, Y in zip(
        contents_Y_hat, contents_Y)]
    # 计算加权风格损失
    styles_l = [style_loss(Y_hat, Y) * style_weight for Y_hat, Y in zip(
        styles_Y_hat, styles_Y_gram)]
    # 计算加权全变分损失
    tv_l = tv_loss(X) * tv_weight
    # 总损失 = 内容损失 + 风格损失 + TV损失
    # 注意：原代码中的"10 * styles_l"可能是笔误，应为"sum(styles_l)"
    l = sum(styles_l) + sum(contents_l) + tv_l
    return contents_l, styles_l, tv_l, l


# =============================================================================
# 步骤8：定义合成图像类（可训练参数）
# =============================================================================
class SynthesizedImage(nn.Module):
    """合成图像类
    
    将合成图像建模为可训练参数（而非神经网络），通过优化直接更新像素值
    """
    def __init__(self, img_shape, **kwargs):
        super(SynthesizedImage, self).__init__(**kwargs)
        # nn.Parameter表示这是可训练参数
        # 初始化为随机噪声，形状与内容图像相同
        self.weight = nn.Parameter(torch.rand(*img_shape))

    def forward(self):
        """前向传播直接返回图像参数"""
        return self.weight
    

def get_inits(X, device, lr, styles_Y):
    """初始化合成图像和优化器
    
    参数:
        X: 内容图像（作为初始值）
        device: 计算设备
        lr: 学习率
        styles_Y: 风格图像特征列表（用于预计算Gram矩阵）
    返回:
        gen_img(): 初始化后的合成图像
        styles_Y_gram: 风格图像的Gram矩阵列表
        trainer: Adam优化器
    """
    # 创建合成图像实例
    gen_img = SynthesizedImage(X.shape).to(device)
    # 将内容图像的数据复制为初始值（通常比随机噪声收敛更快）
    gen_img.weight.data.copy_(X.data)
    # Adam优化器：适合优化图像像素这种高维参数
    trainer = torch.optim.Adam(gen_img.parameters(), lr=lr)
    # 预计算风格图像的Gram矩阵（在训练中保持不变）
    styles_Y_gram = [gram(Y) for Y in styles_Y]
    return gen_img(), styles_Y_gram, trainer


# =============================================================================
# 步骤9：定义训练函数
# =============================================================================
def train(X, contents_Y, styles_Y, device, lr, num_epochs, lr_decay_epoch):
    """训练风格迁移模型
    
    参数:
        X: 内容图像
        contents_Y: 内容图像特征
        styles_Y: 风格图像特征
        device: 计算设备
        lr: 初始学习率
        num_epochs: 训练轮数
        lr_decay_epoch: 学习率衰减周期
    返回:
        X: 训练后的合成图像
    """
    # 初始化
    X, styles_Y_gram, trainer = get_inits(X, device, lr, styles_Y)
    # 学习率调度器：每lr_decay_epoch轮将学习率乘以0.8
    scheduler = torch.optim.lr_scheduler.StepLR(trainer, lr_decay_epoch, 0.8)
    # 可视化动画器
    animator = d2l.Animator(xlabel='epoch', ylabel='loss',
                            xlim=[10, num_epochs],
                            legend=['content', 'style', 'TV'],
                            ncols=2, figsize=(7, 2.5))
    
    for epoch in range(num_epochs):
        trainer.zero_grad()
        
        # 提取合成图像的特征
        contents_Y_hat, styles_Y_hat = extract_features(
            X, content_layers, style_layers)
        
        # 计算损失
        contents_l, styles_l, tv_l, l = compute_loss(
            X, contents_Y_hat, styles_Y_hat, contents_Y, styles_Y_gram)
        
        # 反向传播和优化
        l.backward()
        trainer.step()
        scheduler.step()
        
        # 每10轮更新一次显示
        if (epoch + 1) % 10 == 0:
            animator.axes[1].imshow(postprocess(X))
            animator.add(epoch + 1, [float(sum(contents_l)),
                                     float(sum(styles_l)), float(tv_l)])
    return X


# =============================================================================
# 步骤10：执行风格迁移
# =============================================================================
# 配置设备和图像尺寸
device, image_shape = d2l.try_gpu(), (300, 450)  # 高300，宽450
net = net.to(device)

# 获取内容图像和特征
content_X, contents_Y = get_contents(image_shape, device)
# 获取风格图像特征
_, styles_Y = get_styles(image_shape, device)

# 训练风格迁移
# 参数：学习率0.3，训练500轮，每50轮衰减学习率
output = train(content_X, contents_Y, styles_Y, device, 0.3, 500, 50)

In [ ]:
# =============================================================================
# 神经风格迁移 - 简化执行版本
# =============================================================================
# 本代码是上述完整版本的简化版，直接执行风格迁移流程
# 适用于快速测试或教学演示

%matplotlib inline
import torch
import torchvision
from torch import nn
from d2l import torch as d2l

# 设置图像显示尺寸
d2l.set_figsize()

# =============================================================================
# 步骤1：加载并显示输入图像
# =============================================================================
# 内容图像：提供物体的结构信息（如雷尼尔山峰）
content_img = d2l.Image.open('../img/rainier.jpg')
d2l.plt.figure()  # 创建新Figure，确保图像分开显示
d2l.plt.imshow(content_img);

# 风格图像：提供艺术风格（如秋天的橡树油画风格）
style_img = d2l.Image.open('../img/autumn-oak.jpg')
d2l.plt.figure()  # 关键：再次创建新Figure
d2l.plt.imshow(style_img);


# =============================================================================
# 步骤2：配置训练参数并执行风格迁移
# =============================================================================
# 使用GPU加速（如果可用），设置输出图像尺寸为(高300, 宽450)
device, image_shape = d2l.try_gpu(), (300, 450)

# 将特征提取网络移至GPU
net = net.to(device)

# 获取预处理后的内容图像及其内容特征
content_X, contents_Y = get_contents(image_shape, device)

# 获取风格图像的风格特征（风格图像本身不需要保留）
_, styles_Y = get_styles(image_shape, device)

# 执行训练：学习率0.3，训练500轮，每50轮衰减一次学习率
# 训练完成后，output即为生成的风格迁移图像
output = train(content_X, contents_Y, styles_Y, device, 0.3, 500, 50)